In [22]:
import os
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainer, Seq2SeqTrainingArguments, \
    DataCollatorForSeq2Seq, EarlyStoppingCallback
from datasets import Dataset, DatasetDict
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from translation.main import data

In [3]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name, use_auth_token=os.getenv("HUGGING_FACE_TOKEN"))
model = MBartForConditionalGeneration.from_pretrained(model_name, use_auth_token=os.getenv("HUGGING_FACE_TOKEN"))
tokenizer.src_lang = "kby_Latn"
tokenizer.tgt_lang = "en_XX"

C:\Users\MOPHE\PycharmProjects\Kilba\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
def preprocess_function(examples):
    inputs = [example["kilba"] for example in examples]
    targets = [example["english"] for example in examples]

    # First, force the source language
    tokenizer.src_lang = "kby_Latn"
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Then, force the target language
    tokenizer.tgt_lang = "en_XX"
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [6]:
train_dataset = Dataset.from_dict({
    "kilba": [item["kilba"] for item in data["train"]],
    "english": [item["english"] for item in data["train"]]
})
validation_dataset = Dataset.from_dict({
    "kilba": [item["kilba"] for item in data["validation"]],
    "english": [item["english"] for item in data["validation"]]
})

datasets = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset
})

In [7]:
# Tokenize datasets
tokenized_datasets = datasets.map(
    lambda x: preprocess_function([{key: x[key][i] for key in x} for i in range(len(x["kilba"]))]),
    batched=True
)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

C:\Users\MOPHE\PycharmProjects\Kilba\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=3)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=10,
    num_train_epochs=10,
    predict_with_generate=True,
    logging_dir="./logs",
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[early_stopping_callback]
)

trainer.train()

In [10]:
trainer.save_model("kilba-to-english")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


In [11]:
def translate_kilba_to_english(sentence):
    tokenizer.src_lang = "kby_Latn"

    # Tokenize without src_lang parameter
    inputs = tokenizer(sentence, return_tensors="pt", max_length=128, truncation=True)

    outputs = model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"])
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

In [15]:
kilba_sentence = "nja yah yesu avu baitalami"
# ENGLISH TRANSLATION: "jesus was born in bethlehem"
translated_text = translate_kilba_to_english(kilba_sentence)
print("Translated Text:", translated_text)

Translated Text: i was born in the middle of the world


In [16]:
def calculate_bleu(reference, candidate):
    reference_tokens = reference.split()
    candidate_tokens = candidate.split()
    return sentence_bleu([reference_tokens], candidate_tokens)

In [17]:
reference_sentence = "They then asked, where is he that is born King of the Jews?”"
candidate_sentence = "And he said, Where is he that is born of the sons of Ammon?"

bleu_score = calculate_bleu(reference_sentence, candidate_sentence)
print("BLEU Score:", bleu_score)

BLEU Score: 0.30576902884505114


In [18]:
def calculate_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return scores['rougeL'].fmeasure

In [19]:
rouge_score = calculate_rouge(reference_sentence, candidate_sentence)
print("ROUGE Score:", rouge_score)

ROUGE Score: 0.5925925925925927
